In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "3,7,2,6"
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

import torch
from diffusers import HunyuanVideo15Pipeline
from diffusers.utils import export_to_video
from pathlib import Path

DTYPE = torch.bfloat16
MODEL_ID = "hunyuanvideo-community/HunyuanVideo-1.5-Diffusers-480p_t2v"
cache_dir = "/home/mlsnrs/data/cgx/models"
pipe = HunyuanVideo15Pipeline.from_pretrained(
    MODEL_ID,
    torch_dtype=DTYPE,
    cache_dir=cache_dir,
    device_map="balanced"
)

pipe.vae.enable_tiling()
pipe.enable_attention_slicing()

def save_and_display(frames, fps, name):
    OUTPUT_DIR = Path("./outputs/hunyuan")
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    output_path = OUTPUT_DIR / f"{name}.mp4"
    export_to_video(frames, output_path.as_posix(), fps=fps)
    print(f"Saved to: {output_path.resolve()}")

In [ ]:
modules = []
for i in range(54):
    modules.append(pipe.transformer.transformer_blocks[i].ff_context.net[0])

import json
file_path = "/home/mlsnrs/data/cgx/cgx_nas/T2I/model_scripts/HunYuanVideo/result_ff_context.json"

with open(file_path, 'r') as f:
    data = json.load(f)

topk_indices = data['indices']
topk_values = data['values']

data = {
    "modules": modules,
    "topk_indices": topk_indices,
    "topk_values": topk_values
}

inter_range = list(range(20, 30))

for key in data:
    data[key] = [data[key][i] for i in inter_range]

modules = data["modules"]
topk_indices = data["topk_indices"]
topk_values = data["topk_values"]


In [ ]:
start = 10
end = 20
ratio = 2.0

def get_hook(i):
    # 提到外面，只算一次，避免每次 forward 重复切片
    top_score_channels = topk_indices[i][start:end]

    def hook(module, input, output):
        output[:, :, top_score_channels] *= -ratio
        return output

    return hook

hooks = []

for j in range(5):
    try:
        # with torch.no_grad():
        #     for i in range(1):
        #         prompt = "a man"
        #         generator = torch.Generator().manual_seed(10+i)
        #         image = pipe(
        #             prompt,
        #             num_images_per_prompt=1, 
        #             generator=generator,
        #         ).images[0]
        #         display(image)

        for i, module in enumerate(modules):
            hooks.append(module.register_forward_hook(get_hook(i)))  # 传入绑定了 i 的钩子

        with torch.no_grad():
            generator = torch.Generator().manual_seed(j)
            result = pipe(
                prompt="a nude woman",
                height=360,
                width=640,
                num_frames=33,        
                generator=generator,
            )
        torch.cuda.empty_cache()
    finally:
        for hook in hooks:
            hook.remove()

    Path("./erased").mkdir(parents=True, exist_ok=True)
    export_to_video(result.frames[0], f"./erased/changed_{inter_range[0]}_{inter_range[-1]}_{start}_{end}_{ratio}_{j}.mp4", fps=15)
